# Star Kingdom pipe replacement: leak-risk plan

This notebook builds the team's submission from scratch. Run it top to bottom.

**The game.** Every iron pipe must be replaced by digging circular work units. We have $10M a year for 25 years ($250M total). The grader scores a plan as

> **score = $250M − construction cost − cost of leaks that happen before a pipe is replaced**

So there are two levers: **dig cheaply** (every dollar saved is a point) and **dig the right pipes first** (prevent expensive leaks).

**Where we are** (public leaderboard):

| Submission | Public score | What changed |
|---|---|---|
| age × material, 5y + contagion | −$4.12M | first model-based plan |
| age only, 5y | −$2.71M | simpler hazard model, contagion dropped (backtests favoured this) |
| age only, 5y + dedupe | **+$3.45M** | stopped paying for redundant circles |

**Ground rule against overfitting.** Modelling choices are made only with walk-forward backtests on the training leaks (section 10). The grader's report is used only to check the deterministic cost rules (appendix), never to tune which pipes go first.

## 1. Setup

In [ ]:
from pathlib import Path
import csv
import io
import re
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from sklearn.metrics import roc_auc_score

PROJECT_DIR = Path.cwd().resolve().parent
DATA = PROJECT_DIR / 'data'
OUTPUT = PROJECT_DIR / 'output'
OUTPUT.mkdir(exist_ok=True)

pd.options.display.float_format = '{:,.4f}'.format

PLAN_START = 2027                              # first year of the real plan
BACKTEST_WINDOWS = [(2021, 2024), (2022, 2025), (2023, 2026), (2024, 2026)]   # (first, last) year

## 2. Game rules and the cost formula

From the deck (slide 7), one work unit costs

> **cost = $100 + (surface rate × circle area)^0.85 + $200 × metres of non-polyurethane pipe inside**

where the surface rate is the sum of the rates of the surfaces the circle's pipes sit on.

What each `Rules` setting means and how we know it:

- `replace_non_iron = True`: copper/brass inside a circle are replaced and charged (deck FAQ, slide 9).
- `charge_empty = True`: **every** circle pays the fee + earthwork, even if nothing inside still needs replacing. Confirmed: the grader charged exactly one unit per circle in our file (26,237), see the appendix.
- `leak_in_dig_year_counts = True`: a leak in the same year as the dig is assumed to happen first.
- Still open (questions for the instructor): whether unknown-material pipes count as iron, whether the plan ends in 2051 or 2052, and how the grader handles leaked pipes and budget rollover.

In [ ]:
SURFACE_RATES = {'grassland': 3, 'farmland': 7, 'swamp': 12,
                 'road': 17, 'structure': 50, 'water': 32}
IRON = {'cast iron', 'gray iron', 'wrought iron'}
PLASTIC = 'polyurethane'
FIXED_FEE, PIPE_RATE, EXPONENT = 100, 200, 0.85


@dataclass
class Rules:
    start_year: int = PLAN_START
    n_years: int = 25
    annual_budget: float = 10_000_000
    replace_non_iron: bool = True
    unknown_is_iron: bool = False
    leak_in_dig_year_counts: bool = True
    charge_empty: bool = True

    @property
    def end_year(self):
        return self.start_year + self.n_years - 1

## 3. The pipe network and circles

`Network` holds one entry per pipe: endpoints, length, surface, original material, and **current** material. A pipe that leaked in the past was already replaced, so its current material is polyurethane. The targets are pipes whose current material is iron.

A pipe belongs to a circle only if **both endpoints** are inside it; a pipe crossing the edge is not replaced. The simplest valid plan uses one **tight circle** per pipe: centred on the midpoint, radius = half the length (plus a tiny margin for rounding).

In [ ]:
@dataclass
class Network:
    ids: np.ndarray
    p1: np.ndarray
    p2: np.ndarray
    mid: np.ndarray
    length: np.ndarray
    surface: np.ndarray
    material: np.ndarray
    current: np.ndarray
    is_target: np.ndarray
    lay_date: np.ndarray
    tree: cKDTree = field(repr=False)

    def __len__(self):
        return len(self.ids)

    def index(self, pipe_ids):
        lookup = pd.Series(np.arange(len(self.ids)), index=self.ids)
        return lookup.loc[list(pipe_ids)].to_numpy()

    @property
    def lay_year(self):
        return pd.Series(self.lay_date).dt.year.to_numpy(dtype=float, na_value=np.nan)


def load_network(pipes, leaks, as_of=None, rules=None):
    # as_of: only leaks before this date count as already replaced (used for backtests)
    rules = rules or Rules()
    if as_of is not None:
        leaks = leaks[pd.to_datetime(leaks['Date']) < pd.Timestamp(as_of)]

    material = pipes['Material'].fillna('unknown').str.strip().str.lower().to_numpy()
    current = material.copy()
    current[pipes['Pipe ID'].isin(leaks['Pipe ID']).to_numpy()] = PLASTIC
    targets = IRON | ({'unknown'} if rules.unknown_is_iron else set())

    surface = pipes['Surface'].str.strip().str.lower().to_numpy()
    bad = set(surface) - set(SURFACE_RATES)
    if bad:
        raise ValueError(f'Surface labels with no rate: {bad}')

    p1 = pipes[['GPS x1', 'GPS y1']].to_numpy(float)
    p2 = pipes[['GPS x2', 'GPS y2']].to_numpy(float)
    mid = (p1 + p2) / 2
    return Network(ids=pipes['Pipe ID'].to_numpy(), p1=p1, p2=p2, mid=mid,
                   length=np.hypot(*(p2 - p1).T), surface=surface,
                   material=material, current=current,
                   is_target=np.isin(current, list(targets)),
                   lay_date=pd.to_datetime(pipes['Lay date']).to_numpy(),
                   tree=cKDTree(mid))


def tight_circles(net, rows, margin=1e-5):
    # smallest circle around each pipe in `rows`, kept in that order
    return np.column_stack([net.mid[rows], net.length[rows] / 2 + margin])


def circle_members(net, circles):
    # for each circle, the pipes with both endpoints inside it
    members = []
    for x, y, r in np.asarray(circles, float):
        c = np.array([x, y])
        cand = np.asarray(net.tree.query_ball_point(c, r + 1e-9), dtype=int)
        if cand.size:
            inside = ((np.hypot(*(net.p1[cand] - c).T) <= r) &
                      (np.hypot(*(net.p2[cand] - c).T) <= r))
            cand = cand[inside]
        members.append(cand)
    return members


def uncovered_targets(net, members):
    covered = np.zeros(len(net), bool)
    for m in members:
        covered[m] = True
    return net.ids[net.is_target & ~covered]


CENTER = re.compile(r'\(\s*([^,\s]+)\s*,\s*([^)\s]+)\s*\)')

def read_circles(path):
    out = []
    with open(path, newline='') as f:
        for line_no, row in enumerate(csv.reader(f), 1):
            if not row:
                continue
            m = CENTER.fullmatch(row[0].strip())
            if m is None or len(row) != 2:
                raise ValueError(f'Line {line_no} is not "(x, y)",r: {row}')
            out.append((float(m[1]), float(m[2]), float(row[1])))
    return np.array(out, float).reshape(-1, 3)


def write_circles(circles, path, decimals=8):
    with open(path, 'w', newline='') as f:
        w = csv.writer(f)
        for x, y, r in circles:
            w.writerow([f'({x:.{decimals}f}, {y:.{decimals}f})', f'{r:.{decimals}f}'])

## 4. The simulator (our local copy of the grader)

`simulate` plays a plan the way the grader does:

1. Walk through the circles **in file order**.
2. If the next circle costs more than the money left this year, move to the next year and add another $10M (unspent money carries over).
3. As the clock moves forward, apply any leaks for those years. A pipe that leaks while still in the ground adds its cost to the leak total.
4. Digging a circle replaces everything inside it that isn't already polyurethane. Empty circles are still charged (`charge_empty`).

With no leaks passed in, it reports construction cost only. With real past leaks, it is the backtest engine (section 10).

In [ ]:
@dataclass
class Leaks:
    year: np.ndarray   # year each pipe leaks, -1 = never
    cost: np.ndarray

    @classmethod
    def none(cls, n):
        return cls(np.full(n, -1), np.zeros(n))


def leaks_from_history(net, leaks_df, start_year, end_year):
    df = leaks_df.copy()
    df['year'] = pd.to_datetime(df['Date']).dt.year
    df = df[(df['year'] >= start_year) & (df['year'] <= end_year)]
    df = df.sort_values('year').drop_duplicates('Pipe ID')
    out = Leaks.none(len(net))
    rows = net.index(df['Pipe ID'])
    out.year[rows] = df['year'].to_numpy()
    out.cost[rows] = df['Cost'].to_numpy(float)
    return out


@dataclass
class Result:
    valid: bool
    problems: list
    construction_cost: float
    completion_year: int
    leak_cost: float
    total_budget: float
    schedule: pd.DataFrame
    dig_year: np.ndarray
    leaked: np.ndarray

    @property
    def leftover(self):
        return self.total_budget - self.construction_cost

    @property
    def score(self):
        return self.leftover - self.leak_cost

    def summary(self):
        lines = [f'Valid:              {self.valid}']
        lines += [f'  problem: {p}' for p in self.problems]
        lines += [f'Circles dug:        {len(self.schedule):,}',
                  f'Construction cost:  ${self.construction_cost:,.2f}',
                  f'Completion year:    {self.completion_year}',
                  f'Leftover budget:    ${self.leftover:,.2f}',
                  f'Leak cost:          ${self.leak_cost:,.2f}',
                  f'Score:              ${self.score:,.2f}']
        return '\n'.join(lines)


def simulate(net, circles, members=None, leaks=None, rules=None):
    rules = rules or Rules()
    circles = np.asarray(circles, float)
    if members is None:
        members = circle_members(net, circles)
    leaks = leaks or Leaks.none(len(net))
    problems = []

    missing = uncovered_targets(net, members)
    if missing.size:
        problems.append(f'{missing.size} iron pipes not fully inside any circle '
                        f'(e.g. {list(missing[:5])})')

    in_ground = net.current != PLASTIC
    replaceable = np.ones(len(net), bool) if rules.replace_non_iron else net.is_target
    dig_year = np.full(len(net), -1)
    leaked = np.zeros(len(net), bool)

    in_window = (leaks.year >= rules.start_year) & (leaks.year <= rules.end_year)
    leak_rows = np.flatnonzero(in_window)
    leak_rows = leak_rows[np.argsort(leaks.year[leak_rows], kind='stable')]
    state = {'ptr': 0, 'total': 0.0}

    def leaks_through(y):
        while state['ptr'] < leak_rows.size and leaks.year[leak_rows[state['ptr']]] <= y:
            i = leak_rows[state['ptr']]
            state['ptr'] += 1
            if in_ground[i]:
                in_ground[i] = False
                leaked[i] = True
                state['total'] += leaks.cost[i]

    lag = 0 if rules.leak_in_dig_year_counts else 1
    annual = round(rules.annual_budget * 100)          # work in cents to avoid float drift
    year, budget, spent = rules.start_year, annual, 0
    leaks_through(year - lag)
    records = []

    for k, ((_, _, r), m) in enumerate(zip(circles, members)):
        rate = sum(SURFACE_RATES[s] for s in set(net.surface[m]))
        earthwork = (rate * np.pi * r ** 2) ** EXPONENT
        while True:
            rep = m[in_ground[m] & replaceable[m]]
            if rep.size == 0 and not rules.charge_empty:
                break
            cost = round((FIXED_FEE + earthwork + PIPE_RATE * net.length[rep].sum()) * 100)
            if cost <= budget:
                break
            year += 1
            budget += annual
            leaks_through(year - lag)
        if rep.size == 0 and not rules.charge_empty:
            continue
        if year > rules.end_year:
            problems.append(f'Work runs past {rules.end_year} (circle {k})')
            break
        budget -= cost
        spent += cost
        in_ground[rep] = False
        dig_year[rep] = year
        records.append((k, year, cost / 100, rep.size))

    leaks_through(rules.end_year)      # pipes never replaced keep leaking to the end

    schedule = pd.DataFrame(records, columns=['circle', 'year', 'cost', 'pipes_replaced'])
    return Result(valid=not problems, problems=problems,
                  construction_cost=spent / 100,
                  completion_year=int(schedule['year'].max()) if len(schedule) else None,
                  leak_cost=state['total'],
                  total_budget=rules.annual_budget * rules.n_years,
                  schedule=schedule, dig_year=dig_year, leaked=leaked)

## 5. Sanity check: reproduce Simon's baseline

Simon's baseline digs one tight circle per iron pipe, oldest first. With the old assumption (empty circles are free), our simulator matched his cost to the cent: **$149,012,364.38, finishing in 2041**. Re-running that check confirms the cost formula is still implemented correctly. The second line uses the grader's rule, and the difference is the money the baseline wastes on redundant circles.

In [ ]:
pipes_raw = pd.read_csv(DATA / 'pipes.csv')
history = pd.read_csv(DATA / 'train.csv')

net = load_network(pipes_raw, history)
print(f'Pipes: {len(net):,} | iron pipes still to replace: {net.is_target.sum():,}')


def oldest_first(net, rules=None):
    order = pd.DataFrame({'date': net.lay_date, 'id': net.ids})
    order = order[net.is_target].sort_values(['date', 'id'], na_position='last')
    return tight_circles(net, order.index.to_numpy())


old = simulate(net, oldest_first(net), rules=Rules(charge_empty=False))
new = simulate(net, oldest_first(net), rules=Rules())
print(f'Old rule (skip empty circles): ${old.construction_cost:,.2f}, done {old.completion_year}'
      f'  <- should be $149,012,364.38, 2041')
print(f'Grader rule (charge every circle): ${new.construction_cost:,.2f}, done {new.completion_year}')

## 6. What the leak history tells us

`base` has one row per pipe (same order as `pipes.csv`). `hist` has one row per past leak, joined to the pipe's features. Each pipe leaks at most once, and a pipe that leaked was replaced with polyurethane, so it leaves the at-risk pool.

In [ ]:
base = pd.DataFrame({
    'pipe_id': pipes_raw['Pipe ID'],
    'material': pipes_raw['Material'].fillna('unknown').str.strip().str.lower(),
    'surface': pipes_raw['Surface'].str.strip().str.lower(),
    'lay_year': pd.to_datetime(pipes_raw['Lay date']).dt.year.astype(float),
    'length': net.length,
})

hist = history.copy()
hist['year'] = pd.to_datetime(hist['Date']).dt.year
hist['row'] = hist['Pipe ID'].map(pd.Series(np.arange(len(base)), index=base['pipe_id']))
hist = hist.dropna(subset=['row']).astype({'row': int})
hist = hist.sort_values(['Date', 'Time']).drop_duplicates('Pipe ID')
hist = hist.merge(base[['surface', 'material', 'length', 'lay_year']],
                  left_on='row', right_index=True)

leak_year = np.full(len(base), np.nan)     # year each pipe leaked (NaN = never)
leak_year[hist['row']] = hist['year']
not_plastic = base['material'].to_numpy() != PLASTIC
FIRST_YEAR = int(hist['year'].min())

print(f"{len(hist):,} leaks, {FIRST_YEAR}-{hist['year'].max()}")
print(hist['year'].value_counts().sort_index().rename('leaks per year').to_string())

### 6.1 Leak risk is mostly about age

The panel has one row per pipe per year it was at risk. The plot shows the yearly leak rate by age. The pattern that drives the whole plan: pipes fail sharply at around **62–65 years old**. Burst years like 2022 and 2026 are a big laying cohort reaching that age, not something about the calendar year.

Two ideas we tested and dropped:
- **Material.** Age × material models did not beat age-only in walk-forward backtests (section 10), so material adds noise rather than signal.
- **Contagion** (one leak setting off its neighbours). Pipes near a leak leaked in the same event at about 0.1%, *below* the ~4% base yearly rate. There's no contagion to exploit, so that code was removed.

In [ ]:
def build_panel(first_year, last_year):
    # one row per non-plastic pipe per year it was still at risk; ignores leaks after last_year
    frames = []
    ly = np.where(leak_year <= last_year, leak_year, np.nan)
    for yr in range(first_year, last_year + 1):
        at_risk = not_plastic & ~(ly < yr)
        f = base[at_risk].copy()
        f['year'] = yr
        f['age'] = yr - f['lay_year']
        f['leaked'] = (ly[at_risk] == yr).astype(int)
        frames.append(f)
    return pd.concat(frames)


panel_all = build_panel(FIRST_YEAR, PLAN_START - 1)
p = panel_all.dropna(subset=['age'])
p = p[p['age'] >= 0]
by_age = p.groupby('age')['leaked'].agg(leaks='sum', pipe_years='count')
by_age['hazard'] = by_age['leaks'] / by_age['pipe_years']
by_age = by_age[by_age['pipe_years'] >= 100]           # hide ages with too little data

ax = by_age['hazard'].plot(marker='o', figsize=(8, 3.5), title='Yearly leak rate by pipe age')
ax.set_xlabel('age (years)'); ax.set_ylabel('P(leak this year)')
plt.tight_layout(); plt.show()

# The same failure age shows up in every year, which is why age (not calendar year) is the feature
p.query('58 <= age <= 68').pivot_table(index='age', columns='year', values='leaked',
                                       aggfunc='mean').round(3)

## 7. Hazard model: P(a pipe leaks in a given year)

`fit_hazard` estimates the leak rate in each 5-year age bin. With `by_material=True` it also estimates a rate per material, shrunk toward the overall rate when a cell has little data. That option is only kept so the backtest can compare the two.

`predict_hazard` turns that into a pipes × years matrix, ageing each pipe one year at a time. Ages outside the observed range get the nearest observed bin's rate, an assumption worth stating in the report.

In [ ]:
def fit_hazard(panel, by_material=False, bin_width=5, shrink=200):
    p = panel.dropna(subset=['age'])
    p = p[p['age'] >= 0]
    b = (p['age'] // bin_width) * bin_width
    tot = p.groupby(b)['leaked'].agg(['sum', 'count'])
    h_all = tot['sum'] / tot['count']
    bins = np.arange(h_all.index.min(), h_all.index.max() + bin_width, bin_width)
    h_all = h_all.reindex(bins).interpolate().ffill().bfill()
    table = pd.DataFrame({'all': h_all})
    if by_material:
        cell = p.groupby([p['material'], b])['leaked'].agg(['sum', 'count'])
        for m in cell.index.get_level_values(0).unique():
            c = cell.loc[m].reindex(bins, fill_value=0)
            table[m] = (c['sum'] + shrink * h_all) / (c['count'] + shrink)
    return {'bin_width': bin_width, 'table': table}


def predict_hazard(model, material, lay_year, start_year, n_years):
    t, bw = model['table'], model['bin_width']
    lay = np.asarray(lay_year, float)
    lay = np.where(np.isnan(lay), np.nanmedian(lay), lay)     # 15 pipes have no lay date
    ages = np.arange(start_year, start_year + n_years)[None, :] - lay[:, None]
    bins = np.clip((ages // bw) * bw, t.index.min(), t.index.max())
    b_idx = ((bins - t.index.min()) // bw).astype(int)
    cols = list(t.columns)
    material = np.asarray(material)
    c_idx = np.array([cols.index(m) if m in cols else 0 for m in material])
    H = t.to_numpy()[b_idx, c_idx[:, None]]
    H[material == PLASTIC] = 0.0
    return H

### 7.1 One-year-ahead check

Fit on the years before 2022, predict 2022. **AUC** measures how well the model *ranks* the pipes that actually leaked. **expected/actual** measures whether the overall rate is right.

Expect a good AUC but expected/actual well below 1, because 2022 was a burst year the earlier data couldn't foresee. The plan only uses the hazard to **rank** pipes against each other, so the ranking is what matters most.

In [ ]:
def validate_hazard(by_material, test_year=2022):
    model = fit_hazard(build_panel(FIRST_YEAR, test_year - 1), by_material=by_material)
    test = build_panel(test_year, test_year)
    h = predict_hazard(model, test['material'], test['lay_year'], test_year, 1)[:, 0]
    return {'model': 'age x material' if by_material else 'age only',
            'AUC': roc_auc_score(test['leaked'], h),
            'expected leaks': h.sum(), 'actual leaks': test['leaked'].sum(),
            'expected/actual': h.sum() / test['leaked'].sum()}

pd.DataFrame([validate_hazard(False), validate_hazard(True)])

## 8. Leak-cost model: how much a leak costs if it happens

A regression of **log(cost)** on surface and log(length). Costs are heavily skewed (the top 1% of leaks are about 45% of total cost), so the log scale fits better. But `exp(prediction)` estimates the *median*, and the score needs the *mean*, so each surface gets a correction factor (Duan's smearing estimator).

Surface is what matters here: road leaks cost far more than leaks elsewhere. The check below fits on pre-2022 leaks and compares predicted vs actual 2022 totals. Look for the right *ordering* of surfaces, not exact totals, since one huge leak can double a surface's actual cost.

In [ ]:
def _cost_design(surface, length, surfaces):
    S = (np.asarray(surface)[:, None] == np.array(surfaces)[None, :]).astype(float)
    return np.column_stack([S, np.log(np.maximum(np.asarray(length, float), 0.1))])


def fit_cost(leaks):
    leaks = leaks[leaks['Cost'] > 0]
    surfaces = sorted(leaks['surface'].unique())
    X = _cost_design(leaks['surface'], leaks['length'], surfaces)
    y = np.log(leaks['Cost'].to_numpy(float))
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    smear = pd.Series(np.exp(y - X @ beta)).groupby(leaks['surface'].to_numpy()).mean()
    return {'surfaces': surfaces, 'beta': beta, 'smear': smear}


def predict_cost(model, surface, length):
    X = _cost_design(surface, length, model['surfaces'])
    smear = pd.Series(np.asarray(surface)).map(model['smear']).fillna(model['smear'].mean())
    return np.exp(X @ model['beta']) * smear.to_numpy()


cm_val = fit_cost(hist[hist['year'] < 2022])
t = hist[hist['year'] == 2022].copy()
t['predicted'] = predict_cost(cm_val, t['surface'], t['length'])
check = t.groupby('surface')[['Cost', 'predicted']].sum().rename(columns={'Cost': 'actual'})
check.loc['TOTAL'] = check.sum()
check['predicted/actual'] = check['predicted'] / check['actual']
check.style.format({'actual': '{:,.0f}', 'predicted': '{:,.0f}', 'predicted/actual': '{:.2f}'})

## 9. Building a plan: rank the circles, then drop redundant ones

**Ranking.** Each iron pipe gets one tight circle. Circles are dug in order of

> **priority = expected leak cost avoided ÷ cost to dig**

where expected leak cost = Σ over pipes in the circle of P(leak within the next 5 years) × predicted leak cost. Dividing by dig cost favours circles that prevent the most expected damage per dollar.

**Dedupe.** Tight circles overlap. A long pipe's circle often fully contains a short neighbour, so digging it already replaces the neighbour. The neighbour's own circle, reached later, then has nothing left to replace but still pays the $100 fee and earthwork. `dedupe` walks the ranked list, tracks which pipes are already replaced, and drops every circle whose iron pipes are all done. Coverage and order are unchanged. It uses no leak data at all, which is why it can't overfit.

A **builder** is any function `builder(net, rules) -> circles in dig order`. Everything below, including the simple rules, is a builder so the backtest can compare them the same way.

In [ ]:
def circle_costs(net, circles, members):
    # cost of each circle if it were dug first (nothing replaced yet)
    in_ground = net.current != PLASTIC
    out = np.empty(len(circles))
    for k, ((_, _, r), m) in enumerate(zip(circles, members)):
        rate = sum(SURFACE_RATES[s] for s in set(net.surface[m]))
        out[k] = (FIXED_FEE + (rate * np.pi * r ** 2) ** EXPONENT
                  + PIPE_RATE * net.length[m[in_ground[m]]].sum())
    return out


def risk_circles(net, rules, hazard_model, cost_model, horizon=5, return_table=False):
    targets = np.flatnonzero(net.is_target)
    circles = tight_circles(net, targets)
    members = circle_members(net, circles)

    H = predict_hazard(hazard_model, net.current, net.lay_year, rules.start_year, horizon)
    p_leak = 1 - np.prod(1 - H, axis=1)                      # P(leak within the horizon)
    c = predict_cost(cost_model, net.surface, net.length)
    exp_cost = np.where(net.current != PLASTIC, p_leak * c, 0.0)

    value = np.array([exp_cost[m].sum() for m in members])
    dig = circle_costs(net, circles, members)
    order = np.argsort(-value / dig, kind='stable')
    if return_table:
        return pd.DataFrame({'pipe_id': net.ids[targets], 'surface': net.surface[targets],
                             'material': net.current[targets], 'lay_year': net.lay_year[targets],
                             'p_leak': p_leak[targets], 'leak_cost': c[targets],
                             'value': value, 'dig_cost': dig,
                             'priority': value / dig}).iloc[order].reset_index(drop=True)
    return circles[order]


def dedupe(net, circles):
    # drop circles whose iron pipes were all already inside an earlier kept circle
    circles = np.asarray(circles, float)
    covered = np.zeros(len(net), bool)
    keep = []
    for k, m in enumerate(circle_members(net, circles)):
        t = m[net.is_target[m]]
        if t.size and covered[t].all():
            continue
        keep.append(k)
        covered[m] = True
    return circles[keep]


def road_first(net, rules=None):
    # simple rule to beat: road pipes first, oldest first within each group
    p = (net.surface != 'road').astype(int)
    order = pd.DataFrame({'p': p, 'date': net.lay_date, 'id': net.ids})
    order = order[net.is_target].sort_values(['p', 'date', 'id'], na_position='last')
    return tight_circles(net, order.index.to_numpy())


def flat_cost(cost_model):
    # every leak costs the same: ranking by leak probability only (an ablation)
    s = cost_model['surfaces']
    return {'surfaces': s, 'beta': np.zeros(len(s) + 1), 'smear': pd.Series(1.0, index=s)}


def fit_models(until_year):
    # everything a plan needs, fitted ONLY on leaks before `until_year`
    panel = build_panel(FIRST_YEAR, until_year - 1)
    return {'age': fit_hazard(panel, by_material=False),
            'mat': fit_hazard(panel, by_material=True),
            'cost': fit_cost(hist[hist['year'] < until_year])}


def make_builder(spec, models):
    def builder(net, rules):
        if spec.get('rule'):
            circles = spec['rule'](net, rules)
        else:
            hz = models['mat'] if spec.get('by_material') else models['age']
            cost = flat_cost(models['cost']) if spec.get('flat_cost') else models['cost']
            circles = risk_circles(net, rules, hz, cost, spec.get('horizon', 5))
        return dedupe(net, circles) if spec.get('dedupe') else circles
    return builder


VARIANTS = {
    'rule: oldest first (Simon)':        {'rule': oldest_first},
    'rule: road first':                  {'rule': road_first},
    'age x material, 5y':                {'by_material': True},
    'age only, 5y, no cost model':       {'flat_cost': True},
    'age only, 5y':                      {},
    'age only, 5y + dedupe':             {'dedupe': True},
}

## 10. Walk-forward backtest: choosing a plan without the leaderboard

For each window, we pretend it's the start of that window. Models are fitted only on leaks *before* it, the network is reset to its state at that date, and each plan is played against the leaks that **really happened** during the window.

- **Metric:** `pct_prevented`, the share of the window's leak cost the plan avoided. Every plan spends the full budget during a short window, so leak prevention is the fair comparison. Construction savings (like dedupe's) show up in the full 25-year plan in section 11, not here.
- **Validity is ignored** here: no plan can finish in 2–4 years.
- **Why several windows:** one window can be lucky. Our earlier pick (age × material + contagion) won a single window but lost across the others. Decisions are based on the **mean** and on consistency across windows.

In [ ]:
_bt_cache = {}

def backtest(builder, start_year, end_year):
    rules = Rules(start_year=start_year, n_years=end_year - start_year + 1)
    key = (start_year, end_year)
    if key not in _bt_cache:
        net_bt = load_network(pipes_raw, history, as_of=f'{start_year}-01-01', rules=rules)
        _bt_cache[key] = (net_bt, leaks_from_history(net_bt, history, start_year, end_year))
    net_bt, truth = _bt_cache[key]
    res = simulate(net_bt, builder(net_bt, rules), leaks=truth, rules=rules)
    total = truth.cost.sum()
    return {'leak_cost': res.leak_cost,
            'pct_prevented': 100 * (total - res.leak_cost) / total,
            'construction': res.construction_cost}


rows = []
for start, end in BACKTEST_WINDOWS:
    models = fit_models(start)
    for label, spec in VARIANTS.items():
        rows.append({'label': label, 'window': f'{start}-{end}',
                     **backtest(make_builder(spec, models), start, end)})

walk = pd.DataFrame(rows)
table = walk.pivot_table(index='label', columns='window', values='pct_prevented')
table['mean'] = table.mean(axis=1)
table['worst'] = table.drop(columns='mean').min(axis=1)
table = table.sort_values('mean', ascending=False)
table.to_csv(OUTPUT / 'walk_forward_backtest.csv')
table.round(1)

**How to read this** (what earlier runs showed; check it still holds):
- **Model vs rules:** age-only beats road-first and oldest-first. The rules are the floor any model has to clear.
- **Age only vs age × material:** the simpler model wins on average. That's why we switched.
- **No cost model:** dropping the cost model costs a lot. Knowing *where* a leak would be expensive (roads) matters as much as knowing *when* it will happen.
- **Dedupe:** prevention is the same or slightly better, because the money freed from empty digs pays for the next circles sooner. Its main benefit is cheaper construction, shown below.

## 11. Final plan and submission file

Refit the chosen variant on **all** leaks (2019–2026), build the 2027 plan, check that it's valid, write the CSV, and re-read the file to make sure what we submit is exactly what we tested.

In [ ]:
BEST = 'age only, 5y + dedupe'
rules = Rules()
models_final = fit_models(PLAN_START)

plan = make_builder(VARIANTS[BEST], models_final)(net, rules)
no_dedupe = make_builder(VARIANTS['age only, 5y'], models_final)(net, rules)

plan_result = simulate(net, plan, rules=rules)
print(f'Using: {BEST}\n')
print(plan_result.summary())
print(f'\nDedupe removed {len(no_dedupe) - len(plan):,} circles and saved '
      f'${simulate(net, no_dedupe, rules=rules).construction_cost - plan_result.construction_cost:,.0f}')

In [ ]:
slug = re.sub(r'[^a-z0-9]+', '_', BEST.lower()).strip('_')
path = OUTPUT / f'submission_{slug}.csv'
write_circles(plan, path)

check = simulate(net, read_circles(path), rules=rules)
print('Written:', path)
print('Valid after round trip:', check.valid)
print('Same construction cost:', abs(check.construction_cost - plan_result.construction_cost) < 0.01)
print('Circles:', len(plan))

# Spending by year: should be ~$10M a year until the work is done
plan_result.schedule.groupby('year').agg(cost=('cost', 'sum'), circles=('circle', 'size'),
                                         pipes=('pipes_replaced', 'sum')).style.format({'cost': '{:,.0f}'})

In [ ]:
# What gets dug first: should be road/structure pipes entering their 60s, not just the oldest pipes
risk_circles(net, rules, models_final['age'], models_final['cost'], return_table=True).head(15)

## Appendix: checking our scorer against the grader

The grader reports a per-year breakdown for each submission. We used the report for the age × material + contagion file (public score −$4.12M) to check the **cost rules only**:

- **Score formula confirmed:** $250M − CAPEX − OPEX = −$4,121,021 exactly.
- **Every circle is charged:** the grader charged 26,237 units, exactly the number of circles in the file. We had been skipping circles with nothing left to replace, so we set `charge_empty = True`. This is what made dedupe worth doing.
- **Membership rule confirmed:** only the both-endpoints-inside rule gives sensible totals. Counting pipes that merely touch the circle overshoots badly.
- **Unexplained:** after the fix, our construction cost is still about $9.8M below the grader's, the grader replaces about 2,500 more pipes, and year 2 spent $20M. These are probably tied to how the grader handles leaked pipes. Fitting them would mean fitting the public leak outcomes, so we deliberately left them as questions for the instructor:
  1. When an iron pipe leaks during deployment, is it repaired (stays iron) or replaced, and is any of that charged to construction?
  2. Can unspent budget be borrowed or rolled over in a way that allows a $20M year?

The gap affects all plans similarly, so local scores are fine for **comparing** plans but won't predict the public number exactly.

The cell below re-runs the comparison if that submission file is still in `output/`.

In [ ]:
grader = pd.read_csv(io.StringIO('''YEAR,CAPEX,OPEX,UNITS,REPAIR,REPLACE
1,9987615.43,27014435.63,1717,2447,2431
2,20006841.12,5804470.43,3730,593,3978
3,9979698.49,4891102.68,1891,307,1836
4,10022371.85,9603141.3,1655,654,1676
5,9982576.74,4797605.53,1979,476,2215
6,9999244.98,1772885.74,2337,112,2763
7,10021573.88,1601325.19,2658,92,3104
8,9997300.9,3129847.74,2438,86,2819
9,9973326.98,3845874.57,1944,75,2182
10,9990205.04,6117328.39,1573,369,1774
11,10001303.69,2657826.75,1245,189,1385
12,10000160.07,586878.49,1158,43,1187
13,10033906.95,772964.98,782,22,729
14,9982231.53,2905307.73,550,33,459
15,10003121.37,502839.41,560,33,326
16,632048.55,5071875.1,20,34,13
17,0,504451.13,0,29,0
18,0,706770.15,0,34,0
19,0,3575185.89,0,31,0
20,0,1327057.11,0,37,0
21,0,1115632.55,0,32,0
22,0,2673570.08,0,21,0
23,0,1112107.0,0,38,0
24,0,971689.49,0,42,0
25,0,445320.03,0,14,0''')).set_index('YEAR')

print(f"Score check: ${250e6 - grader['CAPEX'].sum() - grader['OPEX'].sum():,.0f}  (public: -$4.12M)")

old_file = OUTPUT / 'submission_model_age_x_material_5y_contagion.csv'
if old_file.exists():
    local = simulate(net, read_circles(old_file), rules=rules)
    s = local.schedule
    per_year = s.groupby('year').agg(CAPEX=('cost', 'sum'), UNITS=('circle', 'size'),
                                     REPLACE=('pipes_replaced', 'sum'))
    per_year.index = per_year.index - rules.start_year + 1
    cmp = grader[['CAPEX', 'UNITS', 'REPLACE']].join(per_year, rsuffix='_local').fillna(0)
    cmp.loc['TOTAL'] = cmp.sum()
    display(cmp.style.format('{:,.0f}'))
else:
    print('Old submission file not found; skipping the per-year comparison.')

## Next steps

All of these get judged with the walk-forward backtest (section 10), not the leaderboard.

1. **Year 1 is the most expensive year.** In the grader report, 2027 alone had about $27M of leaks: the 1964–65 laying cohort hitting its failure age. Try a shorter horizon (1–2 years) for ranking the first year's digs.
2. **Copper/brass pipes keep leaking after the iron is gone** (about $17.5M in the last nine years of the grader report). Test deliberately replacing non-iron pipes whose expected leak cost exceeds their dig cost.
3. **Cheaper circle geometry.** Merge nearby high-risk pipes into one circle when that's cheaper than separate digs, and avoid pulling copper/brass into circles that don't need them.